In [0]:
from pyspark.sql import functions as F

# Mart TER

In [0]:
# Lecture Silver
df_ter = spark.read.table("transport.silver.regularite_ter")
df_tgv = spark.read.table("transport.silver.regularite_tgv")
df_transilien = spark.read.table("transport.silver.regularite_transilien")
df_gare = spark.read.table("transport.silver.gares")
df_all_gare = spark.read.table("transport.silver.all_gares")

In [0]:
# Mart 1 - TER
mart_ter = (
    df_ter
    .groupBy("region", F.date_trunc("month", "date").alias("mois"))
    .agg(
        F.avg("taux_regul").alias("taux_regul_moyen"),
        F.sum("nb_trains_annules").alias("total_trains_annules"),
        F.sum("nb_trains_programmes").alias("total_trains_programmes"),
        F.sum("nb_trains_retards").alias("total_trains_retards")
    )
)

In [0]:

mart_ter.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transport.gold.mart_regularite_ter")

# Mart TGV

In [0]:
from pyspark.sql.functions import lower, regexp_replace, trim

def normalize_name(col_name):
    return (
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.lower(col_name),
                        "[àâä]", "a"
                    ),
                    "[éèêë]", "e"
                ),
                "[-_/]", " "
            )
        )
    )

In [0]:
df_tgv_unique = (
    df_tgv
    .filter(F.col('service') == 'National')
    .withColumn("gare_depart", normalize_name("gare_depart")) \
    .select("gare_depart")
    .distinct()
    )

df_gare_unique = (
    df_all_gare
    .withColumn("nom_gare", F.lower(F.col("libelle")))
    .withColumn("commune", F.lower(F.col("commune")))
    .withColumn("nom_gare", F.regexp_replace(F.col("nom_gare"), "-", " "))
    .withColumn("nom_gare", normalize_name("nom_gare"))
    .select("nom_gare", "commune", "geo_point")
    .distinct()
    )

In [0]:
df_join = (
    df_tgv_unique
    .join(df_gare_unique, df_tgv_unique.gare_depart == df_gare_unique.nom_gare, how="left")
    )
# df_join.filter(F.col('gare_depart').contains("ville")).display()
df_join.display()

In [0]:
df_gare_unique = df_gare_unique.dropDuplicates(['commune'])

In [0]:
df_join2 = (
    df_join
    .select("gare_depart")
    .filter(F.col("nom_gare").isNull())
    .join(df_gare_unique, df_join.gare_depart == df_gare_unique.commune, how="left")
)

df_join2.display()

In [0]:
df_join3 = (
    df_join2.filter(F.col("nom_gare").isNull())
    .drop("commune", "nom_gare", "geo_point")
    .withColumn("premier_mot", F.split(F.col("gare_depart"), " ")[0])
    .join(df_gare_unique, F.col("premier_mot") == df_gare_unique["commune"], how="left")
    .drop("premier_mot")
)

df_final = (
    df_join.filter(F.col("nom_gare").isNotNull())
    .unionByName(df_join2.filter(F.col("nom_gare").isNotNull()))
    .unionByName(df_join3)
    .withColumn("latitude", 
        F.split(F.col("geo_point"), ", ")[0].cast("double")
    ).withColumn("longitude",
        F.split(F.col("geo_point"), ", ")[1].cast("double")
    )
)

In [0]:
df_final = df_final.withColumn("latitude",
    F.when(F.col("gare_depart") == "bellegarde (ain)", 46.1077)
    .when(F.col("gare_depart") == "barcelona", 41.3792)
    .when(F.col("gare_depart") == "marne la vallee", 48.8574)
    .when(F.col("gare_depart") == "saint etienne chateaucreux", 45.4397)
    .otherwise(F.col("latitude"))
).withColumn("longitude",
    F.when(F.col("gare_depart") == "bellegarde (ain)", 5.8274)
    .when(F.col("gare_depart") == "barcelona", 2.1686)
    .when(F.col("gare_depart") == "marne la vallee", 2.7794)
    .when(F.col("gare_depart") == "saint etienne chateaucreux", 4.4039)
    .otherwise(F.col("longitude"))
)

In [0]:
df_final.display()

In [0]:
df_gares_tgv = df_final.groupBy("gare_depart") \
    .agg(
        F.avg("latitude").alias("latitude"),
        F.avg("longitude").alias("longitude")
    )

In [0]:
df_gares_tgv.display()

In [0]:
(
    df_gares_tgv
    .withColumnRenamed('gare', 'nom_gare_depart')
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("transport.gold.gare_tgv")
)

# __________________________________

In [0]:

# Mart 2 - TGV
mart_tgv = (
    df_tgv
    .groupBy("gare_depart", "gare_arrivee", F.date_trunc("month", "date").alias("mois"))
    .agg(
        F.avg("retard_moyen_tous_trains_arrivee").alias("retard_moyen_arrivee"),
        F.avg("retard_moyen_tous_trains_depart").alias("retard_moyen_depart"),
        F.sum("nb_trains_annules").alias("total_annules"),
        F.avg("prct_retard_causes_externes").alias("prct_cause_externe"),
        F.avg("prct_retard_cause_infrastructure").alias("prct_cause_infrastructure"),
        F.avg("prct_retard_cause_mat_roulant").alias("prct_cause_materiel")
    )
)

In [0]:

mart_tgv.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transport.gold.mart_retards_tgv")
